# Equation différentielle stochastique de Ornstein-Uhlenbeck

## Objectif
Ce notebook a pour but de simuler numériquement l'Équation Différentielle Stochastique (EDS) d'Ornstein-Uhlenbeck (OU) et de vérifier les résultats théoriques obtenus manuellement (Espérance et Variance).

## Rappel du Modèle
D'après nos calculs, l'EDS est définie par :
$$dX_t = a(\theta - X_t)dt + \sigma dB_t$$

Avec :
* $a$ : Vitesse de retour à la moyenne (mean reversion rate).
* $\theta$ : Moyenne de long terme.
* $\sigma$ : Volatilité (dans l'exercice manuscrit, $\sigma=1$).

Les solutions théoriques trouvées sont :
1.  **Espérance :** $\mathbb{E}[X_t] = x_0 e^{-at} + \theta (1 - e^{-at})$
2.  **Variance :** $\mathbb{V}[X_t] = \frac{\sigma^2}{2a} (1 - e^{-2at})$

### Importation et paramètres

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

T = 5.0
N = 1000
dt = T / N
t_space = np.linspace(0, T, N + 1)   # N+1 instants pour N pas

a = 2.0          # vitesse de rappel
theta = 5.0      # moyenne long terme
sigma = 1.0      # volatilite
x0 = 10.0        # depart loin de theta

n_sims = 2000

print(f"dt={dt}, a={a}, theta={theta}, sigma={sigma}")

## Méthode d'Euler-Maruyama

Pour simuler l'EDS sur un ordinateur, nous devons discrétiser le temps. On utilise l'approximation d'Euler-Maruyama :

$$X_{t+1} = X_t + a(\theta - X_t)\Delta t + \sigma \sqrt{\Delta t} Z$$

Où $Z \sim \mathcal{N}(0, 1)$ est une variable aléatoire normale standard.
Cela revient à dire que l'accroissement brownien $dB_t$ est simulé par $\sqrt{dt} \times \text{Gaussienne}$.


### Boucle de simulation 

In [ ]:
# Euler-Maruyama vectorise sur les n_sims trajectoires
X = np.zeros((n_sims, N + 1))
X[:, 0] = x0

for i in range(1, N + 1):
    Z = np.random.normal(0, 1, n_sims)
    X[:, i] = X[:, i-1] + a * (theta - X[:, i-1]) * dt + sigma * np.sqrt(dt) * Z

print("ok")

### Visualisation

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(50):
    plt.plot(t_space, X[i, :], lw=1, alpha=0.3, color='gray')

plt.axhline(theta, color='red', linestyle='--', linewidth=2, label=r'Cible $\theta$')
plt.title("Trajectoires simulées du processus d'Ornstein-Uhlenbeck")
plt.xlabel("Temps $t$")
plt.ylabel("$X_t$")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Vérification Statistique

C'est l'étape cruciale. Nous allons calculer :
1.  La **moyenne empirique** de nos 2000 simulations à chaque instant $t$.
2.  La **variance empirique** de nos 2000 simulations à chaque instant $t$.

Nous les comparerons ensuite aux formules exactes trouvées dans les calculs manuscrits. Si les courbes se superposent, les calculs sont validés.

In [ ]:
empirical_mean = X.mean(axis=0)
empirical_var = X.var(axis=0)

theoretical_mean = x0 * np.exp(-a * t_space) + theta * (1 - np.exp(-a * t_space))
theoretical_var = (sigma**2 / (2 * a)) * (1 - np.exp(-2 * a * t_space))

print(f"erreur max esperance : {np.max(np.abs(empirical_mean - theoretical_mean)):.4f}")
print(f"erreur max variance  : {np.max(np.abs(empirical_var - theoretical_var)):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(t_space, empirical_mean, 'b-', lw=3, alpha=0.6, label='Empirique')
ax1.plot(t_space, theoretical_mean, 'r--', lw=2, label='Théorique')
ax1.set_title("Validation de l'Espérance $\mathbb{E}[X_t]$")
ax1.legend()
ax1.grid(True)

ax2.plot(t_space, empirical_var, 'b-', lw=3, alpha=0.6, label='Empirique')
ax2.plot(t_space, theoretical_var, 'r--', lw=2, label='Théorique')
ax2.axhline(sigma**2 / (2*a), color='green', linestyle=':', label='Variance asymptotique')
ax2.set_title("Validation de la Variance $\mathbb{V}[X_t]$")
ax2.legend()
ax2.grid(True)

plt.show()

## Analyse des résultats

1.  **Sur l'Espérance :**
    On observe que la courbe simulée colle parfaitement à la courbe théorique. Le processus part de $x_0=10$ et décroît exponentiellement vers $\theta=5$. Cela confirme que le terme de drift $a(\theta - X_t)$ fonctionne comme une force de rappel.

2.  **Sur la Variance :**
    Au temps $t=0$, la variance est nulle (car on part tous de $x_0$ de manière déterministe).
    Ensuite, la variance augmente avec le temps car le mouvement brownien disperse les trajectoires.
    **Point clé :** Contrairement au mouvement brownien simple où la variance croît à l'infini ($t$), ici la variance sature et plafonne à $\frac{\sigma^2}{2a} = \frac{1}{4} = 0.25$.
    
Cela confirme la **stationnarité** du processus d'Ornstein-Uhlenbeck sur le long terme.